In [35]:
import pandas as pd
import numpy as np
df = pd.read_csv('D:/Downloads/APT_combined.csv', low_memory=False)

In [36]:
list_features = [
    'bidirectional_stddev_ps',
    'bidirectional_mean_ps',
    'dst2src_duration_ms',
    'src2dst_duration_ms',
    'bidirectional_bytes',
    'bidirectional_packets',
    'bidirectional_duration_ms',
    'src2dst_packets',
    'dst2src_packets',
    'src2dst_bytes',
    'dst2src_bytes',
    'bidirectional_mean_piat_ms',
    'bidirectional_stddev_piat_ms',
    'bidirectional_max_piat_ms',
    'bidirectional_min_piat_ms',
    'src2dst_mean_piat_ms',
    'src2dst_stddev_piat_ms',
    'src2dst_max_piat_ms',
    'src2dst_min_piat_ms',
    'dst2src_mean_piat_ms',
    'dst2src_stddev_piat_ms',
    'dst2src_max_piat_ms',
    'dst2src_min_piat_ms',
    'bidirectional_fin_packets',
    'bidirectional_syn_packets',
    'bidirectional_rst_packets',
    'bidirectional_psh_packets',
    'bidirectional_ack_packets',
    'bidirectional_urg_packets',
    'bidirectional_cwr_packets',
    'bidirectional_ece_packets',
    'time',
    'locate',
    'Stage',
    'Activity'
]

# Lọc DataFrame theo list_features
df = df[list_features]

In [37]:
# Giả sử stage_mapping là từ điển mà bạn đã định nghĩa
action_mapping = {
    'Maintain Access': 0,
    'Encrypted Channel: Symmetric Cryptography': 1,     
    'Data Transfer Size Limits': 2,
    'Remote System Discovery': 3,
    'Exfiltration over C2 channel': 4,
    'Remove Traces': 5,
    'Unsecured Credentials': 6 
}

stage_mapping = {
    'Reconnaissance': 0,     
    'Establish Foothold': 1,
    'Lateral Movement': 2,
    'Data Exfiltration': 3,
    'Cover up': 4
}

In [38]:
df['Activity'] = df['Activity'].map(action_mapping)
df['Stage'] = df['Stage'].map(stage_mapping)

In [39]:
# Đếm số dòng cho mỗi ngày
daily_counts = df.groupby('time').size().reset_index(name='Count')

# In kết quả
print("Số lượng dòng dữ liệu cho mỗi ngày:")
print(daily_counts)

Số lượng dòng dữ liệu cho mỗi ngày:
    time  Count
0     11     28
1     24    131
2     25    291
3     31    162
4     32     45
5     33     62
6     34     38
7     35     32
8     41     18
9     42     20
10    43     24
11    44     20
12    45     18
13    51    969
14    52   3561
15    53   5490
16    54   5835
17    55  11659
18    56   6148
19    61  16083
20    62   6317
21    63   3366
22    64   1420
23    66   1388


In [40]:
# Thêm cột 'defend' với giá trị ban đầu là 'detecting'
daily_counts['defend'] = 'detecting'

In [41]:
# Hàm tính phần trăm thay đổi và gán nhãn
def assign_defend_label(row, prev_count, prev_label):
    if prev_count is None:  # Ngày đầu tiên
        return 'detecting'
    
    current_count = row['Count']
    percent_change = (current_count - prev_count) / prev_count * 100
    
    if percent_change >= 20:
        return 'detecting'
    if percent_change <= -20:
        return 'restricting'
    else:
        return prev_label

In [42]:
# Gán nhãn 'defend' cho từng ngày
prev_count = None
prev_label = 'detecting'

for i, row in daily_counts.iterrows():
    daily_counts.loc[i, 'defend'] = assign_defend_label(row, prev_count, prev_label)
    prev_count = row['Count']
    prev_label = daily_counts.loc[i, 'defend']

In [43]:
# Gộp nhãn 'defend' vào DataFrame gốc
df['defend'] = df['time'].map(daily_counts.set_index('time')['defend'])

In [44]:
# In kết quả số dòng mỗi ngày và nhãn 'defend'
print("Số lượng dòng dữ liệu và nhãn defend cho mỗi ngày:")
print(daily_counts)

Số lượng dòng dữ liệu và nhãn defend cho mỗi ngày:
    time  Count       defend
0     11     28    detecting
1     24    131    detecting
2     25    291    detecting
3     31    162  restricting
4     32     45  restricting
5     33     62    detecting
6     34     38  restricting
7     35     32  restricting
8     41     18  restricting
9     42     20  restricting
10    43     24    detecting
11    44     20    detecting
12    45     18    detecting
13    51    969    detecting
14    52   3561    detecting
15    53   5490    detecting
16    54   5835    detecting
17    55  11659    detecting
18    56   6148  restricting
19    61  16083    detecting
20    62   6317  restricting
21    63   3366  restricting
22    64   1420  restricting
23    66   1388  restricting


In [45]:
# (Tùy chọn) In vài dòng đầu của DataFrame gốc với cột 'defend'
print("\nDataFrame với cột defend (mẫu):")
print(df.head())


DataFrame với cột defend (mẫu):
   bidirectional_stddev_ps  bidirectional_mean_ps  dst2src_duration_ms  \
0              1130.082479             537.944444                 6369   
1                11.547005              67.333333                    0   
2              1128.860447             546.444444                 5302   
3                11.547005              67.333333                    0   
4               185.635924             162.125000                    1   

   src2dst_duration_ms  bidirectional_bytes  bidirectional_packets  \
0                 6370                 9683                     18   
1                    0                  202                      3   
2                 5303                 9836                     18   
3                    1                  202                      3   
4                    2                 1297                      8   

   bidirectional_duration_ms  src2dst_packets  dst2src_packets  src2dst_bytes  \
0                   

In [46]:
# Tổng số dòng trong file
print(f"\nTổng số dòng trong file: {len(df)}")


Tổng số dòng trong file: 63125


In [47]:
df.to_csv('D:/Downloads/APT_combined_defend.csv', index=False)